# DeepSeek 网页搜索自定义工具

使用 `@tool` 将 [DeepSeek Search API](https://api-docs.deepseek.com/zh-cn/api/search.md) 封装为 LangChain 工具，先直接调用，再让 Agent 根据问题发起搜索。

选择项目的 `.venv` 作为 Notebook 内核，并在项目根目录的 `.env` 中配置 `DEEPSEEK_API_KEY`。搜索接口和聊天模型共用这个密钥。

## 1. 定义搜索工具

`query` 支持 `site:`、`lang:` 等搜索语法，`count` 默认返回最多 5 条。请求使用 `need_content=False`，向模型提供标题、链接、摘要和发布时间。

In [1]:
import os
from typing import Annotated

import requests
from dotenv import find_dotenv, load_dotenv
from langchain.tools import tool
from langchain_core.tools import ToolException
from pydantic import BaseModel, Field, StringConstraints

load_dotenv(find_dotenv(usecwd=True))


class DeepSeekSearchInput(BaseModel):
    query: Annotated[str, StringConstraints(strip_whitespace=True, min_length=1, max_length=128)] = Field(
        description="网页搜索关键词，支持 site: 等搜索语法。"
    )
    count: int = Field(default=5, ge=1, le=50, strict=True, description="最多返回的结果条数。")


@tool(args_schema=DeepSeekSearchInput)
def deepseek_web_search(query: str, count: int = 5) -> dict:
    """用 DeepSeek 搜索网页，返回带来源链接的结果，适合查询实时信息及核实事实。"""
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise ToolException("请在项目根目录的 .env 中配置 DEEPSEEK_API_KEY。")

    try:
        response = requests.post(
            "https://api.deepseek.com/v1/tools/search",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"query": query, "count": count, "need_content": False},
            timeout=60,
        )
        response.raise_for_status()
    except requests.HTTPError as exc:
        raise ToolException(f"DeepSeek 搜索失败（HTTP {exc.response.status_code}）。") from exc
    except requests.RequestException as exc:
        raise ToolException("DeepSeek 搜索请求超时或连接失败。") from exc

    try:
        results = response.json()["results"]
        if not isinstance(results, list):
            raise TypeError("results 应为数组")
        return {
            "results": [
                {
                    "title": item["title"],
                    "url": item["url"],
                    "summary": item["summary"] or item["snippet"],
                    "date": item["date"],
                }
                for item in results
            ]
        }
    except (ValueError, KeyError, TypeError) as exc:
        raise ToolException("DeepSeek Search API 返回了无法解析的搜索结果。") from exc


# 让工具错误作为消息返回给 Agent，便于模型说明失败原因。
deepseek_web_search.handle_tool_error = True

## 2. 直接调用工具

In [2]:
from pprint import pprint

search_result = deepseek_web_search.invoke(
    {"query": "DeepSeek 网页搜索 API site:api-docs.deepseek.com", "count": 3}
)
pprint(search_result, sort_dicts=False)

{'results': [{'title': 'DeepSeek 发布全家桶应用 API | DeepSeek API Docs',
              'url': 'https://api-docs.deepseek.com/zh-cn/news/news261113/',
              'summary': 'DeepSeek 全家桶应用 API 已发布：联网搜索、文件输入、阅读链接、模型安全、ASR、TTS '
                         '等，点此查看使用指南\n'
                         '\n'
                         '# DeepSeek 发布全家桶应用 API\n'
                         '\n'
                         '在 DeepSeek '
                         'App（含网页端）的对话服务中，联网搜索、文件输入、阅读链接、语音交互及安全等能力与模型推理相互配合，构成了用户熟悉的对话体验。\n'
                         '...\n'
                         '今天，我们正式开放这套全家桶应用 API：在原有 API '
                         '中开启对应参数，即可获得深度思考搜索、文件输入等对话功能；语音及相关底层能力也已开放独立接口（含 '
                         'OCR、ASR、TTS 等），支持单独或组合调用。开发者可以在自己的应用中轻松实现复杂的对话功能，甚至 '
                         '1:1 复刻出完整的 DeepSeek App。\n'
                         '\n'
                         '## 全面开放应用功能，支持独立调用\n'
                         '\n'
                         '本次更新将 DeepSeek App 的各项交互功能直接整合进原有 '
                     

## 3. 接入 Agent

把 `deepseek_web_search` 传给 `create_agent()`。模型生成搜索参数，LangChain 调用本地工具，再把搜索结果交给模型撰写回答。

In [4]:
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
agent = create_agent(
    model=model,
    tools=[deepseek_web_search],
    system_prompt=(
        "涉及网页资料时，先使用 deepseek_web_search 工具。"
        "根据搜索结果回答，列出引用的网页链接。搜索失败或无结果时如实说明。"
    ),
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="搜索一些未来5天南京的天气")
        ]
    }
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

搜索一些未来5天南京的天气
================================== Ai Message ==================================
Tool Calls:
  deepseek_web_search (call_00_tMryLvDfQyhfTwZBoKDr3201)
 Call ID: call_00_tMryLvDfQyhfTwZBoKDr3201
  Args:
    query: 南京未来5天天气预报
    count: 5
  deepseek_web_search (call_01_UuWHvh547TsHvz5Av1DH5773)
 Call ID: call_01_UuWHvh547TsHvz5Av1DH5773
  Args:
    query: 南京天气预报 一周 气温
    count: 5
================================= Tool Message =================================
Name: deepseek_web_search

{"results": [{"title": "Weather in Nanjing for 10 days — Weather forecast in Nanjing, Jiangsu, China, for 10 days - — Nanjing", "url": "http://yandex.com.am/pogoda/en/details/running/day-2?via=prsw&lon=118.778921&lat=32.060775#1", "summary": "— Nanjing\nmorning: +22°, clear, Feels like 20.\nafternoon: +22°, clear, Feels like 21.\nevening: +21°, clear, Feels like 21.\nnight: +21°, clear, Feels like 21.\n: +20°, m